# SpeechDyn Playbook

A learning guide to the speaker-diarization evaluation platform in this repo: what each
piece does, why it is shaped that way, and how to drive it by hand when something is stuck.

**How to read this.** Chapters 1-3 are orientation. Chapters 4-11 walk the path a single
audio file takes, in the order the bytes actually travel. Chapter 12 is the operator's
toolkit (the manual re-enqueue recipe and its neighbours). Chapters 13-16 are the
extension and debugging material.

**How to run this.** Most code cells are read-only inspection: safe to run any time.
Cells that write to the database or the queue are clearly marked and gated behind a
`DRY_RUN` flag you have to flip yourself.

Cells that only inspect source or contracts need nothing running. Cells that touch the
database need `docker compose up -d`. Cells that enqueue need Redis up and a worker pool
if you want the job to actually execute.

The kernel needs `ipykernel`, which this project does not install by default:

```bash
uv add --dev ipykernel      # once
```

Then select the `.venv` interpreter as the notebook kernel.

---
## 1. The big picture

The product in one sentence: **upload one audio file, run it through several speaker
diarization engines at once, and compare their raw output on a shared timeline.**

Diarization answers *"who spoke when"* - not *what* they said. Each engine returns a list
of `(speaker, start, end)` turns. Engines disagree, and the disagreement is the point:
this is a KPI evaluation tool, so the UI must show exactly what each model produced.

### The life of one upload

```
   browser
      │  POST /upload?models=pyannote,azure-batch   (WAV bytes)
      ▼
   FastAPI  ──► store bytes in the lane(s) the chosen models need
      │        ──► MinIO      (local lane)
      │        ──► Azure Blob (azure lane, only for azure-batch)
      │
      ├──► Postgres: 1 AudioFile row + 1 EvaluationResult row per model (status=queued)
      ├──► Redis/RQ: 1 job per model
      │
      └──► 202-ish ack: {audioFileId, models:[{id, status:"queued"}]}   ← returns immediately

   RQ worker (N independent processes)
      │  run_model(audio_file_id, model_id)
      ├──► lane_for(model_id) → "local" | "azure"
      ├──► pipeline: mark_running → download/SAS → runner.run() → adapter.adapt() → mark_done
      └──► writes status / payload / timing into THAT model's own row

   browser (polling every FRONTEND_POLL_INTERVAL_MS)
      │  GET /evaluations/{id}       → status + segments per model, straight from Postgres
      └─ GET /evaluations/{id}/audio → API proxies the bytes back for playback + waveform
```

Nothing about that flow is synthetic. There is no fake progress bar; a model is `queued`,
`running`, `done`, or `failed` because a row in Postgres says so.

### The four rules that constrain every change

These are not style preferences. Most of the code's shape follows from them.

**1. Raw model output never crosses a service boundary.**
Every engine emits a different native shape: pyannote gives Python objects, DiariZen gives
RTTM text, the NVIDIA NIM gives a word list with speaker tags, Azure gives a JSON
transcription. Each model has a `runner.py` (executes, returns native output untouched)
and an `adapter.py` (the *only* code allowed to understand that shape). The API, the
database, and the frontend see one unified contract and nothing else.

**2. Segments are never merged or cleaned up.**
One segment per turn the engine reported, including sub-second gaps. Coalescing adjacent
same-speaker segments would hide exactly the behaviour we are here to measure.

**3. Nothing is fabricated.**
No synthetic progress, no synthetic waveform, no canned timing. Status and timing come
from the DB; the waveform is decoded from the real uploaded audio in the browser.

**4. Two storage lanes, chosen per model, never mixed.**
`local` → MinIO. `azure` → Azure Blob. No model belongs to both, and no code copies bytes
from one store to the other. `apps/background_worker/lanes.py` is the single source of
truth.

---
## 2. Bootstrap the notebook

Run this once. Everything below assumes the repo root is the working directory, because
every import in the project is absolute (`from packages.config...`, `from apps...`).

In [ ]:
import os, sys, subprocess, json, textwrap
from pathlib import Path

REPO = Path("/home/dgx-ai-factory/Evals/speech-diar")
os.chdir(REPO)
sys.path.insert(0, str(REPO))

def show(path, start=None, end=None):
    "Print a file (or a slice of it) with line numbers."
    lines = (REPO / path).read_text().splitlines()
    lo = (start - 1) if start else 0
    hi = end if end else len(lines)
    for i, line in enumerate(lines[lo:hi], start=lo + 1):
        print(f"{i:4}  {line}")

def sh(cmd):
    "Run a shell command from the repo root and print its output."
    out = subprocess.run(cmd, shell=True, cwd=REPO, capture_output=True, text=True)
    print(out.stdout or "", end="")
    if out.stderr:
        print("[stderr]", out.stderr, end="")

print("cwd:", os.getcwd())

### Are the services up?

The platform needs three containers (Postgres, Redis, MinIO) plus, for the GPU engines,
whatever model containers are running under `deploy/`. Check before you debug anything:
half of all "the model is stuck" reports are a container that is not running.

In [ ]:
sh("docker compose ps --format 'table {{.Service}}\t{{.Status}}\t{{.Ports}}'")
print()
sh("docker ps --format 'table {{.Names}}\t{{.Status}}' | grep -Ei 'nim|nemo|diarizen|speaker' || echo '(no model containers running)'")

---
## 3. Tooling and layout

Everything Python is managed by `uv`. You never activate a virtualenv; you prefix commands
with `uv run`.

| Command | What it does |
|---|---|
| `uv sync` | Python 3.12 env: api + worker + dev deps. **Not** the heavy model deps. |
| `uv sync --extra models` | Additionally installs torch / pyannote.audio / whisperx. CUDA hosts only. |
| `docker compose up -d` | Postgres :5432, Redis :6379, MinIO :9010 (console :9099). |
| `uv run honcho start` | API + worker pool together, from the `Procfile`. Ctrl+C stops both. |
| `uv run uvicorn apps.backend_api.main:app --reload --port 8010` | API alone. Note **8010**, not 8000. |
| `./scripts/run_workers.sh` | Worker pool alone: `WORKER_CONCURRENCY` processes. |
| `uv run pytest` | Full backend suite. Needs **no** Docker services. |
| `cd apps/frontend && npm run dev` | Vite on :5173, expects the API on 127.0.0.1:8010. |

Ports are unusual on purpose. MinIO's S3 API sits on **9010** rather than the standard
9000, because on the DGX host port 9000 is owned by the Parakeet NIM container's HTTP API.

### Where things live

```
apps/
  backend_api/          FastAPI: main.py, routers/{upload,evaluations,models}.py, dependencies.py
  background_worker/    the job side
    queue_app.py        the RQ Queue object (shared by API and worker)
    worker.py           run_model(audio_file_id, model_id) — the RQ task; dispatches by lane
    lanes.py            LANE_MAP: model id -> "local" | "azure"
    pipelines/          local_pipeline.py, azure_pipeline.py, _common.py (status helpers)
    models/             one folder per engine: runner.py + adapter.py; __init__.py = REGISTRY
  frontend/             React + Vite; src/adapters/ mirrors the backend adapter split
packages/
  config/settings.py    get_settings() — the ONLY place env vars are read
  database/             models.py (ORM), session.py (engine, SessionLocal, init_db)
  storage/              s3_client.py (MinIO lane), azure_blob.py (Azure lane)
  shared_contracts/     schemas.py (Python) + types.ts (TypeScript) — the wire format
deploy/<model>/         GPU containers: docker-compose + server.py + <model>_up.sh
tests/                  conftest.py + the suite
```

---
## 4. Configuration: one door in

**Rule: no code reads `os.environ` directly. Ever.** Every service imports `get_settings()`
from `packages/config/settings.py`, which is a Pydantic `BaseSettings` loaded from `.env`
at the repo root and cached with `@lru_cache`.

The payoff: moving to staging, prod, or the DGX box is a `.env` edit, never a code change.
The cost you should know about: because `get_settings()` is `lru_cache`d, changing `.env`
does **not** affect an already-running process. Restart the API and the workers.

In [ ]:
from packages.config.settings import get_settings
s = get_settings()

for label, value in [
    ("database_url",            s.database_url),
    ("redis_url",               s.redis_url),
    ("worker_concurrency",      s.worker_concurrency),
    ("queue_job_timeout_sec",   s.queue_job_timeout_sec),
    ("s3_endpoint_url",         s.s3_endpoint_url),
    ("s3_bucket",               s.s3_bucket),
    ("diarization_device",      s.diarization_device),
    ("nim_str_grpc",            s.nim_str_grpc),
    ("nim_ofl_grpc",            s.nim_ofl_grpc),
    ("nemo_clustering_url",     s.nemo_clustering_url),
    ("diarizen_url",            s.diarizen_url),
    ("azure_speech_region",     s.azure_speech_region),
    ("azure_batch_job_timeout", s.azure_batch_job_timeout_sec),
]:
    print(f"{label:24} {value}")

### One setting worth understanding

`QUEUE_JOB_TIMEOUT_SEC` (default 3600) is the queue-level ceiling on how long any single
job may run. RQ's own library default is **180 seconds**, which is shorter than several
per-model timeouts (`AZURE_BATCH_JOB_TIMEOUT_SEC` and `NEMO_CLUSTERING_TIMEOUT_SEC` are
both 1800). Without setting `default_timeout` on the Queue, RQ would kill those jobs long
before their own timeout ever got a chance to fire, and you would spend an afternoon
debugging a model that was actually fine.

So: **`QUEUE_JOB_TIMEOUT_SEC` must stay >= the largest per-model timeout.**

That is also why the manual re-enqueue recipe passes `job_timeout=1800` explicitly - more
on that in chapter 12.

In [ ]:
show("apps/background_worker/queue_app.py")

---
## 5. The contract: the spine of the system

If you learn one file in this repo, learn `packages/shared_contracts/schemas.py`. It is the
thing every other layer agrees on.

The core shape is deliberately tiny:

```python
DiarizationSegment:  spk: int   # zero-based speaker index, stable within one run
                     s:   float # start, seconds
                     e:   float # end, seconds
```

A `DiarizationModelRun` is one model's whole output over one file: identity (`id`, `name`,
`short`, `description`), the segments, `num_spk`, plus the live state (`status`, `error`,
`started_at`, `finished_at`, `processing_ms`).

A `DiarizationEvaluation` is what the frontend renders: the audio file id, its duration,
and a list of model runs.

**The contract lives in three files that must stay in sync:**

1. `packages/shared_contracts/schemas.py` (Python, source of truth)
2. `packages/shared_contracts/types.ts` (TypeScript)
3. `apps/frontend/src/types/diarization.ts` (the frontend's working copy)

Change one, change all three. The wire format is **camelCase** (`numSpk`, `audioFileId`);
Python attributes are snake_case with camelCase aliases, handled by `ContractModel`'s
`alias_generator=to_camel`. That is why every route is declared with
`response_model_by_alias=True`.

In [ ]:
from packages.shared_contracts.schemas import (
    DiarizationSegment, DiarizationModelRun, DiarizationEvaluation, normalize_model_run
)

run = DiarizationModelRun(
    id="demo", name="Demo Model", short="Demo", description="a hand-built run",
    segs=[
        DiarizationSegment(spk=1, s=4.0, e=6.2),   # deliberately out of order
        DiarizationSegment(spk=0, s=0.0, e=3.5),
        DiarizationSegment(spk=0, s=6.4, e=8.0),
    ],
)

print("before normalize:  num_spk =", run.num_spk, " order =", [seg.s for seg in run.segs])
run = normalize_model_run(run)
print("after  normalize:  num_spk =", run.num_spk, " order =", [seg.s for seg in run.segs])
print()
print("wire format (camelCase, what the browser actually receives):")
print(json.dumps(run.model_dump(by_alias=True, mode="json"), indent=2))

### `normalize_model_run` and what it is forbidden from doing

```python
def normalize_model_run(run):
    segs = sorted(run.segs, key=lambda seg: seg.s)
    num_spk = max((seg.spk for seg in segs), default=-1) + 1
    return run.model_copy(update={"segs": segs, "num_spk": num_spk})
```

It sorts, and it recounts speakers. That is the whole function, and that is the whole
point. It may **never** merge or drop a segment the model actually produced (rule 2). Every
adapter's output passes through it before being persisted, so `num_spk` can never drift
away from the segments and the timeline never receives unsorted input.

Notice the count is `max(spk) + 1`, not `len(set(spk))`. Adapters are therefore responsible
for emitting **dense, zero-based** speaker indices. Look at how they do it: each one keeps a
`speaker_index: dict` and calls `setdefault(native_label, len(speaker_index))`, which
re-bases whatever the engine called its speakers (`"SPEAKER_07"`, `speaker_tag=3`, `"B"`)
into `0, 1, 2...` **by order of first appearance**. Get that wrong and `num_spk` inflates.

---
## 6. The database: three tables

`packages/database/models.py`. Postgres in production and dev; SQLite in-memory in tests.

- **`users`** - id, email. There is no real auth yet. `init_db()` seeds exactly one user,
  `dev@example.com`, and `get_current_user` in `dependencies.py` returns it for every
  request. The table already has the shape real auth will need, so swapping it in later
  changes no caller.

- **`audio_files`** - the upload. `duration_sec`, `filename`, and the lane keys:
  `s3_key` (MinIO) and `blob_key` / `blob_url` (Azure). **A row only carries the key(s) for
  the lane(s) actually used.** Upload a file for `pyannote` only and `blob_key` stays NULL.
  `upload_ms` is the client-perceived upload time, PATCHed in by the browser afterwards.

- **`evaluation_results`** - **one row per (audio_file, model)**. This is the table you will
  spend your debugging life in. It owns that model's `status`
  (`queued|running|done|failed`), its `error`, its `payload` (the `DiarizationModelRun` as
  JSON), and its timing (`queued_at`, `started_at`, `finished_at`, `processing_ms`).

The `payload` column stores the **unified contract**, never a model's native output. The
database is downstream of the adapter, always.

Schema management is `Base.metadata.create_all()` at API startup. There is no migration
tool at this stage: if you change a column, drop and recreate the local database.

In [ ]:
show("packages/database/models.py", 40, 75)

### Read the live database

This is your bread-and-butter inspection query. Needs Postgres up.

In [ ]:
from packages.database.session import SessionLocal
from packages.database.models import AudioFile, EvaluationResult

with SessionLocal() as db:
    files = db.query(AudioFile).order_by(AudioFile.id.desc()).limit(5).all()
    for f in files:
        lanes = ", ".join(x for x in [
            "minio" if f.s3_key else None,
            "blob" if f.blob_key else None,
        ] if x)
        print(f"\naudio_file #{f.id}  {f.filename}  {f.duration_sec:.1f}s  lanes=[{lanes}]")
        rows = db.query(EvaluationResult).filter_by(audio_file_id=f.id).order_by(EvaluationResult.id).all()
        for r in rows:
            n = len((r.payload or {}).get("segs", []))
            ms = f"{r.processing_ms}ms" if r.processing_ms else "-"
            err = f"  !! {r.error[:60]}" if r.error else ""
            print(f"    {r.model_id:24} {r.status:8} segs={n:<5} {ms:>9}{err}")

---
## 7. Storage: two lanes that never touch

This is the rule people most often break by accident, so it has its own chapter.

| | **local lane** | **azure lane** |
|---|---|---|
| Store | MinIO / S3 (`packages/storage/s3_client.py`) | Azure Blob (`packages/storage/azure_blob.py`) |
| Pipeline | `pipelines/local_pipeline.py` | `pipelines/azure_pipeline.py` |
| Models | pyannote, whisperx, azure (real-time), all NIM/NeMo/3D-Speaker/DiariZen | **only** `azure-batch` |
| How the engine gets the audio | downloaded to a temp file on disk | a read **SAS URL**; Azure fetches it itself |

Why two? `azure-batch` is a URL-based API: you hand Azure a link and it pulls the audio.
Everything else wants a real file path on disk. Rather than pretend one store serves both,
the platform gives each lane its own store and forbids crossing.

`azure_pipeline.py` imports Azure Blob and never MinIO. `local_pipeline.py` imports MinIO
and never Azure. Even `pipelines/_common.py`, which they share, deliberately has **no
storage imports at all**, so that importing it can never blur the boundary.

One neat detail on the Azure side: blobs are keyed by **content hash**, not by
`audio_file.id`. Re-upload the same file while testing and it reuses the blob already
staged there instead of paying for the upload again.

In [ ]:
from apps.background_worker.lanes import LANE_MAP, lane_for, split_by_lane

for model_id, lane in LANE_MAP.items():
    print(f"{model_id:24} -> {lane}")

print()
print("split_by_lane(['pyannote', 'azure-batch', 'diarizen', 'not-a-model']):")
print(" ", split_by_lane(["pyannote", "azure-batch", "diarizen", "not-a-model"]))
print("  ^ unknown ids are silently dropped from both lists; the upload route logs and skips them")

### The audio proxy

The browser **never** talks to MinIO or Azure directly. `GET /evaluations/{id}/audio`
streams the bytes back through the API, from whichever lane owns the file. Two reasons:
one origin (so no CORS setup and no storage credentials anywhere near the browser), and a
single URL that serves both playback and the client-side waveform decode.

It honours HTTP `Range` requests, so seeking into a 90-minute file does not re-download
everything before the seek point. `_parse_range` handles the single-range case that
browsers actually send, and both `_iter_s3_object` and `_iter_blob` **resume from the last
delivered byte** on a mid-transfer read failure (capped at 5 retries) rather than
truncating playback.

---
## 8. `POST /upload`: the fan-out

`apps/backend_api/routers/upload.py`. Read it in order; it is the clearest 60 lines in the
codebase.

1. **Read the bytes, get the duration.** `_wav_duration_sec` parses the WAV header, no
   transcode, no temp file. It is WAV-only by design; anything else gets a `415`.

   There is a real bug guarded here. Some encoders (streamed or live-recorded WAV) write a
   placeholder `data` chunk size, and `wave.getnframes()` trusts it blindly, so a 30-second
   clip can report as 40 minutes. The fix: cap the frame count by what could physically fit
   in the uploaded bytes.

2. **Split the requested models by lane.** `split_by_lane` gives `(local_ids, azure_ids)`.
   Unknown ids are logged and dropped. No valid ids at all → `422`. Azure ids requested but
   Azure Blob not configured → `422` with a message that names the missing env vars, rather
   than a failure eight seconds later inside a worker.

3. **Write the bytes to each selected lane's store, independently.** Local ids → MinIO under
   `audio/{id}.wav`. Azure ids → Blob under `uploads/{sha256[:32]}.wav`. If you selected
   only local models, the Azure branch never executes and no Azure credential is ever
   touched.

4. **One `EvaluationResult` row per model**, status `queued`.

5. **One RQ job per model**: `queue.enqueue(run_model, audio_file.id, model_id)`.

6. **Return immediately** with `{audioFileId, models: [...]}`. The browser polls from here.

Step 5 is the design decision that makes the whole tool usable. If one job ran all the
models, a 20-minute `azure-batch` run would hide a 4-second NIM result behind it. Instead
each model gets its own job and its own row, and a slow model never blocks a fast one.

In [ ]:
show("apps/backend_api/routers/upload.py", 96, 145)

---
## 9. The queue: RQ, Redis, and N processes

`apps/background_worker/queue_app.py` defines **one** `Queue` object named `diarization`.
Both sides import it: the API to `enqueue()`, the worker to consume.

An RQ job is just a serialized reference: *"call this function with these args"*. Enqueuing
`run_model` does not run it and does not import the engine. It pushes
`(apps.background_worker.worker.run_model, 19, "diarizen")` into Redis and returns. Some
worker process picks it up, imports the function by path, and calls it. That is why the
worker must be able to import the same code the API can.

### Why N processes instead of one forking worker

`scripts/run_workers.sh` starts `WORKER_CONCURRENCY` **separate OS processes**, each running
`rq worker --worker-class rq.SimpleWorker diarization`.

`SimpleWorker` never forks (forking is unsafe for RQ on macOS, and unpleasant with CUDA
contexts anywhere), which means a single SimpleWorker handles exactly one job at a time.
So the parallelism has to come from the process count. Redis hands each queued job to
exactly one worker, so N processes = N models running concurrently.

**The practical consequence:** if `WORKER_CONCURRENCY=1` and you select five models, they
run strictly one after another, and four of them sit in `queued` looking suspiciously
stuck. They are not stuck. They are queued. Check `WORKER_CONCURRENCY` in `.env` before you
debug anything else.

The script also self-heals: it kills any worker PIDs its own previous run recorded in
`.worker_pool.pids`, and it runs `rq empty diarization` on startup to drop stale queued
jobs. Note that second one - **starting the worker pool clears the queue.** If you enqueue
a job and *then* start the workers, the job is gone.

### Inspect the queue

Read-only. Needs Redis up.

In [ ]:
from apps.background_worker.queue_app import queue, redis_conn
from rq.registry import StartedJobRegistry, FailedJobRegistry, FinishedJobRegistry

print("redis:", redis_conn.ping() and "up")
print(f"queue '{queue.name}'  default_timeout={queue._default_timeout}s")
print()
print(f"  queued   : {queue.count}")
print(f"  started  : {len(StartedJobRegistry(queue=queue))}")
print(f"  finished : {len(FinishedJobRegistry(queue=queue))}")
print(f"  failed   : {len(FailedJobRegistry(queue=queue))}")

pending = queue.jobs[:10]
if pending:
    print("\npending jobs:")
    for j in pending:
        print(f"  {j.id[:8]}  {j.func_name}{j.args}  timeout={j.timeout}")

failed = FailedJobRegistry(queue=queue)
if len(failed):
    print("\nfailed jobs (RQ-level, e.g. timeout kill or import error):")
    for jid in failed.get_job_ids()[:5]:
        j = queue.fetch_job(jid)
        if j:
            print(f"  {j.id[:8]}  {j.func_name}{j.args}")
            print(f"     {(j.exc_info or '').strip().splitlines()[-1] if j.exc_info else ''}")

### RQ-level failure vs model-level failure

Two different things, and telling them apart saves a lot of time.

- **Model-level failure**: the job ran, the engine raised, the pipeline caught it and called
  `mark_failed`. The `EvaluationResult` row says `status='failed'` with a readable `error`.
  This is the normal, healthy path for a broken model.

- **RQ-level failure**: the job never got to finish. Timeout kill, worker crash, or an
  import error at job start. The row is stranded in `running` (or even `queued`) **forever**,
  because nothing was alive to write `failed` into it. The evidence lives in RQ's
  `FailedJobRegistry`, not in Postgres.

A row stuck in `running` with no worker alive is the signature of the second case. That is
exactly what the manual re-enqueue recipe in chapter 12 exists to clean up.

---
## 10. The worker and the two pipelines

`worker.py` is deliberately tiny. It is a dispatcher and nothing else:

```python
def run_model(audio_file_id: int, model_id: str) -> None:
    lane = lane_for(model_id)
    if lane == "local":
        run_local_model(audio_file_id, model_id)
    elif lane == "azure":
        run_azure_model(audio_file_id, model_id)
    else:
        logger.warning("No lane mapped for model %r ... skipping", model_id)
```

It never touches storage itself. Note the arguments: `(audio_file_id, model_id)` - two
scalars, no ORM objects, no file handles. RQ has to serialize them into Redis, so a job's
arguments must be plain data. The worker re-opens its own DB session on the other side.

### `local_pipeline.run_local_model`

```
open a DB session
  find the EvaluationResult row  ── missing? log + return (stale job, no row to write to)
  find the model in REGISTRY     ── missing? mark_failed
  audio_file.s3_key present?     ── missing? mark_failed("not stored in MinIO for the local lane")
  mark_running                                       ← status='running', started_at=now
  with tempfile.NamedTemporaryFile() as scratch:     ← auto-deleted, even on exception
      download_to(s3_key, scratch)
      raw = model.runner.run(scratch.name)           ← native output
      run = normalize_model_run(model.adapter.adapt(raw))   ← contract
  mark_done(payload=run.model_dump(by_alias=True))   ← camelCase JSON into Postgres
```

`NotImplementedError` is caught separately and turned into a clean "no runner implementation
yet" - that is how the `whisperx` stub reports honestly instead of crashing. Any other
exception is logged with a traceback and written into the row's `error`.

### `azure_pipeline.run_azure_model`

Same skeleton, different middle: instead of downloading, it builds a **read SAS URL** and
hands that to the runner. There is no local file in this lane at all.

Which creates a problem worth knowing about. Azure Batch diarization rejects stereo audio,
and rejects WAVs whose header frame count disagrees with the actual payload - and it reports
both as the same opaque `"InvalidData - the recordings URI contains invalid data"`. In the
local lane you could patch the temp file. Here the blob itself is what Azure fetches. So
`_sas_url_for_azure_batch` repairs the blob **once**, into a derived `-fixed.wav` key
(downmix to mono, rewrite the header with real chunk sizes), and SASes off that. Blobs that
are already well-formed are passed through untouched.

Finally, `mark_done` for this lane prefers **Azure's own reported duration**
(`adapter.processing_ms(raw)`) over the worker's wall clock, because the wall clock includes
polling overhead that has nothing to do with the model.

In [ ]:
show("apps/background_worker/pipelines/_common.py", 20, 50)

Read those three helpers carefully, because there is a subtlety hiding in them that will
bite you in chapter 12:

**`mark_done` writes `status`, `finished_at`, `processing_ms`, and `payload`. It does not
clear `error`.**

So if a model failed once and you re-run it successfully without clearing the row, the row
ends up `status='done'` **with a stale error string still attached** - and
`GET /evaluations/{id}` copies that `error` straight into the response. The UI will show a
completed model wearing an error from a previous life. That is why the re-enqueue recipe
resets `error = None` explicitly.

---
## 11. Models: the runner / adapter split

This is the heart of the design. Every engine is exactly two objects.

```python
class ModelRunner(ABC, Generic[TRaw]):
    model_id: str          # stable id, e.g. "diarizen"
    available: bool = True # False for stubs → GET /models lists but disables them
    def run(self, audio_path, params=None) -> TRaw:
        "Execute the engine. Return its NATIVE output, whatever shape that is."

class ModelAdapter(ABC, Generic[TRaw]):
    name: str; short: str; description: str   # static display metadata
    def adapt(self, raw: TRaw) -> DiarizationModelRun:
        "Translate. Must be pure."
    def audio_duration_sec(self, raw) -> float | None: ...   # optional
    def processing_ms(self, raw) -> int | None: ...          # optional; engine's own timing

class DiarizationModel(Generic[TRaw]):
    def process(self, audio_path, params=None) -> DiarizationModelRun:
        raw = self.runner.run(audio_path, params)
        return normalize_model_run(self.adapter.adapt(raw))
```

`TRaw` is a type variable: the runner's native output type, whatever it happens to be. Only
the matching adapter is allowed to know what it is. The type system enforces the boundary
that rule 1 describes in prose.

`REGISTRY` in `models/__init__.py` pairs them up, keyed by `model_id`. That is the only
place the full list exists.

In [ ]:
from apps.background_worker.models import REGISTRY
from apps.background_worker.lanes import lane_for

print(f"{'model_id':26} {'lane':7} {'avail':6} name")
print("-" * 90)
for mid, m in REGISTRY.items():
    print(f"{mid:26} {str(lane_for(mid)):7} {str(m.available):6} {m.adapter.name}")

### The same idea, four native shapes

Look at what each adapter has to swallow. This is why the split exists.

| Model | Native output the runner returns |
|---|---|
| `pyannote` | `[{"start": 0.5, "end": 3.2, "label": "SPEAKER_00"}, ...]` from a pyannote `Annotation` |
| `diarizen` | raw **RTTM text**: `SPEAKER file 1 0.50 2.70 <NA> <NA> spk_0 <NA> <NA>` |
| `nim-sortformer-str` | a **word list**: `[{"speaker_tag": 0, "start_ms": 500, "end_ms": 780, "word": "hi"}, ...]` |
| `azure-batch` | Azure's transcription JSON, with per-phrase speaker fields |

Three completely different jobs to reach the same three numbers. The NIM adapter has the
most interesting one: the engine tags **individual words** with a speaker, so the adapter
groups *consecutive same-tag words* into one segment per contiguous speaker run. The RTTM
adapter has it easiest: one segment per line.

And note what none of them do: **merge**. Two adjacent RTTM lines with the same speaker stay
two segments, because that is what the model said.

In [ ]:
show("apps/background_worker/models/diarizen/adapter.py")

### Adapters are pure, so you can test them on a napkin

`adapt()` takes native data and returns a contract object. No I/O, no network, no GPU. You
can exercise any adapter in this notebook right now, with no services running and no model
container up - which is exactly what `tests/test_adapters.py` does.

In [ ]:
from apps.background_worker.models.diarizen.adapter import DiarizenAdapter
from packages.shared_contracts.schemas import normalize_model_run

# The exact text a real DiariZen container would return, hand-written.
fake_rttm = {
    "audio_duration_sec": 12.0,
    "rttm": "\n".join([
        "SPEAKER meeting 1 0.50 2.20 <NA> <NA> spk_1 <NA> <NA>",
        "SPEAKER meeting 1 3.00 1.50 <NA> <NA> spk_0 <NA> <NA>",
        "SPEAKER meeting 1 4.60 0.30 <NA> <NA> spk_0 <NA> <NA>",
        "SPEAKER meeting 1 5.10 2.90 <NA> <NA> spk_1 <NA> <NA>",
    ]),
}

out = normalize_model_run(DiarizenAdapter().adapt(fake_rttm))
print(f"num_spk: {out.num_spk}")
for seg in out.segs:
    print(f"  spk={seg.spk}  {seg.s:5.2f} -> {seg.e:5.2f}")

print()
print("Two things just happened, and both are the whole point:")
print()
print("1. RE-BASING. The RTTM label 'spk_1' appears FIRST, so it becomes contract index 0,")
print("   and 'spk_0' becomes index 1. The contract's speaker indices are positional, not")
print("   the engine's names. Every adapter does this the same way:")
print("       speaker_index.setdefault(label, len(speaker_index))")
print()
print("2. NO MERGING. The RTTM lines at 3.00 and 4.60 are both 'spk_0' and nearly adjacent")
print("   (a 0.10s gap). They stayed TWO segments, because that is what the model reported.")
print("   Coalescing them would erase exactly the turn-taking behaviour we exist to measure.")

### Honest stubs

`whisperx` is registered but its `run()` raises `NotImplementedError` and its `available`
flag is `False`. `GET /models` passes that flag through, so the UI **lists it and disables
it** rather than hiding it or, worse, faking output. That is rule 3 applied to the model
registry: an unfinished engine is visible and honest, not absent.

If it is somehow enqueued anyway, `local_pipeline` catches the `NotImplementedError` and
writes an honest `failed` with "no runner implementation yet".

(`pyannote` was a stub too, and is now fully implemented - the registry table you printed
above is the authoritative answer to which engines are live, since it reads `available`
straight off each runner. Prose goes stale; that cell does not.)

### Lazy imports, and why they matter

`pyannote/runner.py` imports `torch` and `pyannote.audio` **inside** `_load_pipeline()`,
never at module scope. `REGISTRY` is built on import in every API process, every worker, and
every test run - if those heavy CUDA-only packages were imported at module level, you could
not even start the API on a laptop without `uv sync --extra models`. The same discipline
applies to any new heavy engine you add.

`_load_pipeline` also caches the loaded pipeline in a module global. Since each SimpleWorker
process is long-lived, the weights are loaded once per worker process, not once per job.

---
## 12. The operator's toolkit

The chapter you actually opened this notebook for.

### 12.1 Manually re-enqueue a stuck or failed model

This is the recipe from the top of the session, unpacked. Use it when a row is stranded in
`running` (worker was killed mid-job), or when a model `failed` for a reason you have since
fixed (container was down, `.env` was wrong, timeout was too short).

```python
with SessionLocal() as db:
    for model_id in ['nim-sortformer-str', 'azure-batch']:
        r = db.query(EvaluationResult).filter(
            EvaluationResult.audio_file_id == 19,
            EvaluationResult.model_id == model_id,
        ).one()
        r.status = 'queued'      # back to the start state
        r.error = None           # ← the important one; mark_done never clears this
        r.started_at = None      # so the next run's timing is honest
        r.finished_at = None
    db.commit()

for model_id in ['nim-sortformer-str', 'azure-batch']:
    queue.enqueue(run_model, 19, model_id, job_timeout=1800)
```

**Why each line is there:**

- **Reset the row, don't create one.** The pipelines call `get_result_row(...)` and *return
  immediately* if it is missing ("dropping stale job"). The row is the job's write target;
  no row, no work. You are recycling the existing row, not making a new one.

- **`status = 'queued'`** is mostly cosmetic (the pipeline calls `mark_running` the moment it
  picks the job up) but it makes the UI honest during the gap between enqueue and pickup.

- **`error = None` is not cosmetic.** `mark_done` does not clear `error`. Skip this line, and
  a successful re-run leaves the row `done` with the *old* failure text still attached, which
  `GET /evaluations/{id}` faithfully hands to the frontend. You get a green model displaying
  a red error and you doubt your sanity.

- **`started_at = None` / `finished_at = None`** because `mark_done` computes
  `processing_ms = finished - started`. A stale `started_at` from the previous attempt would
  produce garbage timing.

- **Note `payload` is *not* cleared.** `mark_done` always overwrites it, so there is no stale
  read. Clearing it too is harmless if you prefer the row visibly empty while it re-runs.

- **`queue.enqueue(run_model, 19, model_id, job_timeout=1800)`** - `run_model` is the
  function *reference*; RQ serializes the path plus the two scalar args into Redis.
  `job_timeout` is per-job and overrides the queue default. It is only meaningful **below**
  `QUEUE_JOB_TIMEOUT_SEC`; passing 1800 here just says "this one job may take up to 30
  minutes", which matters for `azure-batch` and the NeMo clustering models.

**Preconditions.** Redis must be up (or `enqueue` throws) and a worker pool must already be
running (or the job sits queued forever). And remember: `run_workers.sh` runs
`rq empty diarization` on startup, so **start the workers first, then enqueue.**

Below is the same recipe, parameterised and safe. It refuses to run unless you set
`DRY_RUN = False`, and it prints exactly what it would do first.

In [ ]:
# ------------------------- EDIT THESE -------------------------
AUDIO_FILE_ID = 19
MODEL_IDS     = ["nim-sortformer-str", "azure-batch"]
JOB_TIMEOUT   = 1800
DRY_RUN       = True     # ← flip to False to actually reset + enqueue
# --------------------------------------------------------------

from packages.database.session import SessionLocal
from packages.database.models import EvaluationResult
from apps.background_worker.queue_app import queue
from apps.background_worker.worker import run_model
from apps.background_worker.lanes import lane_for

with SessionLocal() as db:
    rows = []
    for model_id in MODEL_IDS:
        r = (
            db.query(EvaluationResult)
            .filter(
                EvaluationResult.audio_file_id == AUDIO_FILE_ID,
                EvaluationResult.model_id == model_id,
            )
            .one_or_none()
        )
        if r is None:
            print(f"!! no EvaluationResult row for ({AUDIO_FILE_ID}, {model_id}) — "
                  f"the pipeline would drop this job. Nothing to re-enqueue.")
            continue
        print(f"{model_id:24} lane={lane_for(model_id):6} current status={r.status:8} "
              f"error={(r.error or '-')[:40]}")
        rows.append(r)

    if DRY_RUN:
        print(f"\nDRY_RUN — would reset {len(rows)} row(s) and enqueue {len(rows)} job(s).")
    else:
        for r in rows:
            r.status = "queued"
            r.error = None
            r.started_at = None
            r.finished_at = None
        db.commit()
        for r in rows:
            job = queue.enqueue(run_model, AUDIO_FILE_ID, r.model_id, job_timeout=JOB_TIMEOUT)
            print(f"enqueued {r.model_id:24} job={job.id[:8]} timeout={JOB_TIMEOUT}s")
        print("\nre-enqueued. Watch it with the polling cell below.")

### 12.2 Watch the run

Poll the same rows until they settle. This reads the database, which is exactly what the
frontend does.

In [ ]:
import time

def watch(audio_file_id, model_ids=None, timeout_sec=120, interval=2.0):
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        with SessionLocal() as db:
            q = db.query(EvaluationResult).filter_by(audio_file_id=audio_file_id)
            if model_ids:
                q = q.filter(EvaluationResult.model_id.in_(model_ids))
            rows = q.order_by(EvaluationResult.id).all()
            line = "  ".join(f"{r.model_id}={r.status}" for r in rows)
            print(f"\r{line}", end="", flush=True)
            if all(r.status in ("done", "failed") for r in rows) and rows:
                print("\n")
                for r in rows:
                    n = len((r.payload or {}).get("segs", []))
                    print(f"{r.model_id:24} {r.status:8} segs={n:<5} {r.processing_ms or '-'}ms")
                    if r.error:
                        print(f"    error: {r.error[:200]}")
                return
        time.sleep(interval)
    print("\ntimed out waiting — still running, or no worker is consuming the queue")

watch(AUDIO_FILE_ID, MODEL_IDS, timeout_sec=60)

### 12.3 Run an engine directly, with no queue and no database

The fastest way to answer "is the model itself broken, or is the plumbing broken?". This
calls the runner and adapter in-process. It needs the engine's container to be up (or its
weights available), but it needs no Redis, no Postgres, and no worker.

If this works and the queued job does not, the bug is in the plumbing. If this fails too,
the bug is in the engine or its config.

In [ ]:
# Needs the model's container running. Point AUDIO_PATH at any mono WAV.
AUDIO_PATH = "/path/to/sample.wav"
MODEL_ID   = "diarizen"

if Path(AUDIO_PATH).exists():
    model = REGISTRY[MODEL_ID]

    raw = model.runner.run(AUDIO_PATH)                 # native output — the ONLY place you see it
    print("native output type:", type(raw).__name__)
    print(str(raw)[:400], "...\n")

    run = normalize_model_run(model.adapter.adapt(raw))  # contract
    print(f"{run.name}: {run.num_spk} speakers, {len(run.segs)} segments")
    for seg in run.segs[:8]:
        print(f"  spk={seg.spk}  {seg.s:6.2f} -> {seg.e:6.2f}")
else:
    print(f"set AUDIO_PATH to a real WAV file (got: {AUDIO_PATH})")

### 12.4 Other things you will need

**Drain the queue** (drop everything still waiting; does not touch rows already `running`):

```bash
uv run rq empty diarization
```

**See what the workers are doing**, live:

```bash
uv run rq info --url redis://localhost:6379/0
```

**Find every stranded row** - `running` with nothing alive to finish it. These are the rows
the re-enqueue recipe is for.

In [ ]:
from datetime import datetime, timezone, timedelta

with SessionLocal() as db:
    stale = (
        db.query(EvaluationResult)
        .filter(EvaluationResult.status == "running")
        .order_by(EvaluationResult.started_at)
        .all()
    )
    if not stale:
        print("no rows in 'running' — nothing stranded")
    for r in stale:
        age = "?"
        if r.started_at:
            started = r.started_at if r.started_at.tzinfo else r.started_at.replace(tzinfo=timezone.utc)
            age = f"{(datetime.now(timezone.utc) - started).total_seconds() / 60:.1f} min"
        print(f"audio_file={r.audio_file_id:<5} {r.model_id:24} running for {age}")
    print("\nIf a row has been 'running' far longer than that model's timeout and no worker")
    print("is alive, its job died without writing back. Re-enqueue it with 12.1.")

**Check a model container is reachable** before blaming the app. Each HTTP engine exposes
its own base URL from settings; the NIM engines are gRPC only.

In [ ]:
import socket, httpx

def probe_http(name, url):
    try:
        r = httpx.get(url, timeout=3.0)
        print(f"{name:22} {url:32} HTTP {r.status_code}")
    except Exception as e:
        print(f"{name:22} {url:32} UNREACHABLE ({type(e).__name__})")

def probe_grpc(name, hostport):
    host, port = hostport.split(":")
    try:
        with socket.create_connection((host, int(port)), timeout=3.0):
            print(f"{name:22} {hostport:32} open")
    except Exception as e:
        print(f"{name:22} {hostport:32} UNREACHABLE ({type(e).__name__})")

probe_http("nemo-clustering",      s.nemo_clustering_url + "/health")
probe_http("3d-speaker",           s.speaker3d_clustering_url + "/health")
probe_http("diarizen",             s.diarizen_url + "/health")
probe_grpc("nim-sortformer-str",   s.nim_str_grpc)
probe_grpc("nim-sortformer-ofl",   s.nim_ofl_grpc)

**Drive the API end to end** without the browser. `POST /upload`, then poll. This is the
whole product in fifteen lines.

In [ ]:
API = "http://127.0.0.1:8010"

# A tiny valid silent WAV — the same helper the test suite uses.
import io, wave
def make_wav_bytes(duration_sec=5.0, framerate=16000):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(framerate)
        w.writeframes(b"\x00\x00" * int(duration_sec * framerate))
    return buf.getvalue()

try:
    print("GET /models:")
    for m in httpx.get(f"{API}/models", timeout=5).json():
        print(f"  {m['id']:24} available={m['available']}")

    # Uncomment to actually upload and fan out to a model:
    # resp = httpx.post(
    #     f"{API}/upload",
    #     params={"models": "diarizen"},
    #     files={"file": ("sample.wav", make_wav_bytes(5.0), "audio/wav")},
    #     timeout=60,
    # )
    # ack = resp.json()
    # print("\nupload ack:", ack)
    # watch(ack["audioFileId"], timeout_sec=120)
except Exception as e:
    print(f"API not reachable at {API} ({type(e).__name__}) — start it with: uv run honcho start")

---
## 13. The test harness

`uv run pytest` runs the whole backend suite **with no Docker services up**. The harness
stands in for real infrastructure, which is why the suite is fast and why you should reach
for it before reaching for a live run.

- **`db_session_factory` / `db_session`** - a fresh **in-memory SQLite** schema per test, dev
  user already seeded. Never the real Postgres.
- **`fake_queue`** - an RQ `Queue` backed by `fakeredis`. `enqueue()` records the job; nothing
  runs it. No worker process starts.
- **`client`** - a `TestClient` with `get_db` overridden to SQLite and `upload.queue`
  monkeypatched to `fake_queue`. Deliberately **not** used as `with TestClient(app)`, because
  the context-manager form fires the app's `startup` event, which calls `init_db()` against
  the real, configured Postgres. Route handlers work fine without startup, since every DB
  access goes through the overridden dependency.
- **`make_wav_bytes(duration_sec, framerate)`** - a tiny valid silent mono WAV, real header,
  real frames.
- **Live tests** - `@pytest.mark.live` is skipped unless `RUN_LIVE_TESTS=1`. Those hit real
  Azure and cost real quota.

One fixture detail worth understanding, because it looks like a hack and is not: the SQLite
session uses `expire_on_commit=False`. SQLite has no timezone-aware column type, so a
post-commit reload would silently hand back **naive** datetimes and break the pipelines'
aware-datetime arithmetic. Keeping the in-memory values sidesteps a SQLite-only artifact
without contorting production code, which always runs on Postgres `TIMESTAMPTZ`.

The files map onto the layers: `test_adapters.py`, `test_contracts.py`, `test_lanes.py`,
`test_worker_dispatch.py` (unit); `test_api_*.py`, `test_pipelines.py` (integration);
`test_live_azure_batch.py` (live).

In [ ]:
sh("uv run pytest -q 2>&1 | tail -15")

---
## 14. Adding a diarization model

Four steps. If you need a fifth, the design is fighting you.

**1. `apps/background_worker/models/<name>/runner.py`** - execute the engine, return native
output untouched. Import heavy deps *lazily*, inside the function, never at module scope.

```python
class MyRunner(ModelRunner[MyRawOutput]):
    model_id = "my-model"

    def run(self, audio_path: str, params: dict | None = None) -> MyRawOutput:
        settings = get_settings()                    # never os.environ
        with open(audio_path, "rb") as f:
            r = httpx.post(f"{settings.my_model_url}/diarize",
                           files={"file": (audio_path, f, "audio/wav")},
                           timeout=settings.my_model_timeout_sec)
        r.raise_for_status()
        return r.json()                              # native shape, untouched
```

**2. `apps/background_worker/models/<name>/adapter.py`** - translate to `DiarizationModelRun`.
One segment per turn the engine reported. Re-base speaker labels to dense zero-based indices
by first appearance. Never merge.

```python
class MyAdapter(ModelAdapter[MyRawOutput]):
    name = "My Model 1.0"
    short = "MyModel"
    description = "One line, shown on the model card"

    def adapt(self, raw: MyRawOutput) -> DiarizationModelRun:
        speaker_index: dict[str, int] = {}
        segs = []
        for turn in raw["turns"]:
            spk = speaker_index.setdefault(turn["speaker"], len(speaker_index))
            segs.append(DiarizationSegment(spk=spk, s=turn["start"], e=turn["end"]))
        return DiarizationModelRun(
            id="my-model", name=self.name, short=self.short, description=self.description,
            segs=segs, num_spk=len(speaker_index),
        )
```

**3. Register in `models/__init__.py`** - one line in `REGISTRY`:
`DiarizationModel(MyRunner(), MyAdapter())`.

**4. Add to `LANE_MAP` in `lanes.py`** - `"my-model": "local"`.

That is the whole integration. `GET /models` picks it up from `REGISTRY` automatically, the
frontend lists it, `POST /upload` accepts it, `run_model` dispatches it. No other file
changes.

If the engine needs new config, add fields to `Settings` **and** to `.env.example`. If it
needs a GPU container, chapter 15.

**Write a test for the adapter** while you are there. It is a pure function over a dict, so
the test is ten lines and it will catch the speaker-rebasing bug you are about to write.

---
## 15. Deploying a GPU model

**GPU inference runs outside the API and the worker.** The app talks to it over a network
endpoint (gRPC or HTTP); it does not load weights in-process. That keeps the worker
processes light, lets several workers share one GPU container, and means a model crash never
takes the API down.

The exception that proves the rule is `pyannote`, which loads in-process and caches the
pipeline in a module global. That is why its imports are lazy: so a laptop without CUDA can
still run the API.

Each engine gets a folder under `deploy/<model>/`:

```
deploy/<model>/
  docker-compose.yml          # or docker-compose.<model>.yml — gpus: all, host ports, volumes
  <model>.env.example         # committed template: ports, image/tag, runtime options
  <model>.env                 # local secrets — gitignored
  <model>_up.sh               # pull/start, health check, print the endpoint URL
  server.py                   # for the custom containers: the thin HTTP wrapper
```

Four engines currently live there. Three of them ship a `server.py`, because no turnkey NIM
exists for them: the container is a bare model, and `server.py` is the small HTTP wrapper
that gives it a `/diarize` endpoint the runner can call.

| Folder | Engine | Endpoint |
|---|---|---|
| `deploy/parakeet/` | Parakeet-Sortformer NIM (streaming + offline) | gRPC :50051 / :50052 |
| `deploy/nemo-clustering/` | MarbleNet VAD + TitaNet + spectral clustering | HTTP :9020 |
| `deploy/3d-speaker-clustering/` | FSMN VAD + CAM++ embeddings, ASR-free | HTTP :9021 |
| `deploy/diarizen/` | WavLM-Large + Conformer EEND + clustering | HTTP :9022 |

A gRPC-only detail worth remembering: the Parakeet NIM's plain HTTP
`/v1/audio/transcriptions` route returns flat text with **no speaker information**. Speaker
tags are only exposed on the Riva gRPC ASR API, as word-level `speaker_tag` fields. That is
why those two runners speak gRPC and the others speak HTTP.

In [ ]:
sh("ls -1 deploy/*/ | head -40")

---
## 16. Failure modes, and what they actually mean

| Symptom | Most likely cause | Where to look |
|---|---|---|
| Models sit in `queued` forever | No worker running, or `WORKER_CONCURRENCY=1` and they are genuinely behind another job | `rq info`; `.env` |
| A model I enqueued vanished | `run_workers.sh` runs `rq empty diarization` on startup. Start workers **first**, enqueue second | `scripts/run_workers.sh` |
| Row stuck in `running`, nothing happening | Job died without writing back (timeout kill, worker `kill -9`, import error) | RQ `FailedJobRegistry`; then re-enqueue (12.1) |
| Job killed at ~180s despite a longer model timeout | `QUEUE_JOB_TIMEOUT_SEC` unset, so RQ's 180s library default applies | `queue_app.py`, `.env` |
| Model shows `done` **and** an error string | The row was re-run without clearing `error`; `mark_done` never clears it | The re-enqueue recipe (12.1) |
| `415 Only WAV uploads are supported` | The file is not WAV, or its header is unreadable | `_wav_duration_sec` in `upload.py` |
| Duration is absurdly long (30s clip → 40 min) | Encoder wrote a placeholder `data` chunk size. Already guarded on upload; if you see it elsewhere, that code is trusting `getnframes()` | `_wav_duration_sec` |
| `azure-batch`: "InvalidData - the recordings URI contains invalid data" | Stereo audio, or a WAV header whose frame count disagrees with the payload | `_sas_url_for_azure_batch` repairs both into a `-fixed.wav` blob |
| `422 azure-batch needs Azure Blob configured` | `AZURE_STORAGE_ACCOUNT_NAME` / `_KEY` missing | `.env` |
| "Audio was not stored in MinIO for the local lane" | The upload only staged the Azure lane; you enqueued a local model against it afterwards | `upload.py` writes only the lanes the *requested* models need |
| A model failed with "no runner implementation yet" | It is a stub (`whisperx`; `available=False`) | `models/<name>/runner.py` |
| `.env` change had no effect | `get_settings()` is `@lru_cache`d. Restart the API and the workers | `packages/config/settings.py` |
| Frontend shows nothing but the API works | Vite expects the API on 127.0.0.1:**8010**, and `CORS_ALLOW_ORIGINS` must include :5173 | `.env` |
| Timeline shows fewer segments than the model produced | Something merged. That is a bug, not a feature | the adapter; rule 2 |

### The debugging order that works

1. **Is the container up?** `docker compose ps`, plus the model container probes in 12.4.
2. **Is a worker consuming?** `rq info`. If `queued > 0` and no worker is busy, that is your answer.
3. **What does the row say?** Chapter 6's inspection query. `failed` with an error is a *good*
   outcome: the system told you the truth.
4. **Is the engine or the plumbing broken?** Run the runner directly (12.3). That bisects it
   in one step.
5. **Only then read logs.**

---
## 17. The mental model, compressed

If you remember nothing else:

- **One contract.** `DiarizationModelRun` is what every layer speaks. Native model output
  exists only between a runner and its adapter, and nowhere else in the system.

- **One row per (audio file, model).** That row is the truth about status, timing, error, and
  output. The frontend is a view of it. Debugging is reading it.

- **One job per model.** Independent status, independent failure, independent timing. A slow
  model never blocks a fast one. Concurrency is N separate SimpleWorker processes, not
  forking.

- **Two lanes, never crossed.** MinIO for everything local; Azure Blob for `azure-batch`
  alone. `LANE_MAP` decides, nothing else.

- **One config door.** `get_settings()`. Environments differ by `.env`, not by code.

- **Never fabricate, never merge.** Every segment on the timeline is a turn a model actually
  reported. Every status is a row in Postgres. The value of the whole tool is that you can
  trust what it shows you.

### Where to read next, in order

1. `packages/shared_contracts/schemas.py` - the contract.
2. `apps/backend_api/routers/upload.py` - the fan-out.
3. `apps/background_worker/pipelines/local_pipeline.py` - the loop that does the work.
4. `apps/background_worker/models/base_model.py` - the abstraction the models satisfy.
5. `apps/background_worker/models/diarizen/adapter.py` - the smallest complete example.